# Molecular Rotational Analysis: Center of Mass, Inertia Tensor, and Dipole Transformation

This notebook demonstrates how to compute key molecular properties from Cartesian coordinates and atomic masses:

1. **Center of Mass (COM)** — the mass-weighted geometric centroid
2. **Moment of Inertia Tensor** — a 3×3 matrix encoding how mass is distributed in 3D
3. **Principal Moments & Rotational Constants** — eigenvalues of the inertia tensor, convertible to spectroscopic constants
4. **Dipole Moment Transformation** — how the molecular dipole shifts when the origin changes

The example molecule is **4-bromonitrosobenzene cation** (C₆H₄BrNO₂⁺), an aromatic cation with a nitro group and a bromine substituent. All coordinates are in **Ångströms** and masses in **atomic mass units (amu)**.

---
## 0. Imports and Physical Constants

In [2]:
import numpy as np

# ── Physical Constants ─────────────────────────────────────────────────────────
H          = 6.62607015e-34       # Planck's constant, J·s
C_LIGHT    = 29979245800.0        # Speed of light in vacuum, cm/s
AMU_TO_KG  = 1.66053906660e-27   # 1 amu → kg
ANG_TO_M   = 1e-10               # 1 Å → m
BOHR_TO_ANG = 0.52917721090      # 1 bohr → Å  (Psi4 convention)
ANG_TO_BOHR = 1.0 / BOHR_TO_ANG  # 1 Å → bohr

TOTAL_CHARGE = 1.0               # Net charge of the cation (e)

---
## 1. Molecule Definition

We hard-code the geometry of 4-bromonitrosobenzene cation directly as NumPy arrays.

Each row is one atom: `[x, y, z]` in Ångströms. Masses are in amu (isotope-specific values).

In [3]:
# Symbols, Cartesian coordinates (Å), and isotopic masses (amu)
# Geometry from a Psi4 optimization — NOT yet shifted to the center of mass

symbols = np.array(['C','C','C','C','C','H','H','H','H','N','O','O','C','H','BR'])

coords_unshifted = np.array([
    [ 0.029499810,  1.339725920,  0.068177230],  # C1
    [ 1.434832780,  1.286679670,  0.006353130],  # C2
    [ 2.111790240,  0.051061170, -0.005441380],  # C3
    [ 1.445066360, -1.137200580,  0.031165830],  # C4
    [-0.687931710,  0.168222200,  0.109953140],  # C5
    [-0.471269970,  2.298396660,  0.078113550],  # H6
    [ 2.027327830,  2.196517280, -0.032206240],  # H7
    [ 1.989665260, -2.076432170,  0.023184940],  # H8
    [-1.771634800,  0.180405470,  0.158196320],  # H9
    [ 3.586358950,  0.050972920, -0.067452860],  # N10
    [ 4.147117590, -1.059660970, -0.088078490],  # O11
    [ 4.144978590,  1.163909510, -0.090108230],  # O12
    [-0.023611770, -1.145827910,  0.083534830],  # C13
    [-0.436749960, -1.872473640,  0.788895760],  # H14
    [-0.535916380, -1.869721950, -1.740786710],  # Br15
])

masses = np.array([
    12.000000000,   # C
    12.000000000,   # C
    12.000000000,   # C
    12.000000000,   # C
    12.000000000,   # C
     1.007825032,   # H
     1.007825032,   # H
     1.007825032,   # H
     1.007825032,   # H
    14.003074004,   # N
    15.994914620,   # O
    15.994914620,   # O
    12.000000000,   # C
     1.007825032,   # H
    78.918337600,   # Br
])

# Dipole moment in the ORIGINAL (unshifted) frame, from Psi4 (atomic units: e·a₀)
dipole_unshifted_au = np.array([-0.1208851, -0.2497723, -0.0089932])

# Reference values from Psi4 for the COM-shifted geometry (for verification)
rot_psi4   = [0.06088, 0.01841, 0.01526]        # cm⁻¹
dipole_shifted_psi4_au = np.array([-1.9326733, 1.0386462, 1.2694640])  # e·a₀

print(f"Molecule: {' '.join(symbols)}")
print(f"Number of atoms: {len(symbols)}")
print(f"Total molecular mass: {masses.sum():.6f} amu")

Molecule: C C C C C H H H H N O O C H BR
Number of atoms: 15
Total molecular mass: 201.950366 amu


---
## 2. Center of Mass

The **center of mass** (COM) is the mass-weighted average position of all atoms:

$$\mathbf{R}_{\text{COM}} = \frac{\sum_i m_i \, \mathbf{r}_i}{\sum_i m_i} = \frac{\sum_i m_i \, \mathbf{r}_i}{M}$$

where $m_i$ is the mass of atom $i$, $\mathbf{r}_i = (x_i, y_i, z_i)$ is its position vector, and $M = \sum_i m_i$ is the total mass.

Once the COM is found, we **shift** all coordinates so that the COM sits at the origin:

$$\mathbf{r}_i' = \mathbf{r}_i - \mathbf{R}_{\text{COM}}$$

This COM frame is required before computing the inertia tensor.

In [4]:
# Total mass
total_mass = np.sum(masses)

# Center of mass: sum of (mass * position) / total mass
# masses[:, np.newaxis] broadcasts masses to shape (N,1) so we can multiply elementwise
com = np.sum(masses[:, np.newaxis] * coords_unshifted, axis=0) / total_mass

# Shift all coordinates to COM frame
coords_shifted = coords_unshifted - com

print(f"Total molecular mass:  {total_mass:.6f} amu")
print(f"Center of mass (Å):    [{com[0]:.8f},  {com[1]:.8f},  {com[2]:.8f}]")
print()
print("Verification — COM of shifted coordinates (should be ~0,0,0):")
com_check = np.sum(masses[:, np.newaxis] * coords_shifted, axis=0) / total_mass
print(f"  [{com_check[0]:.2e},  {com_check[1]:.2e},  {com_check[2]:.2e}]")

Total molecular mass:  201.950366 amu
Center of mass (Å):    [0.95875707,  -0.68180169,  -0.67653045]

Verification — COM of shifted coordinates (should be ~0,0,0):
  [-2.11e-16,  7.04e-17,  7.04e-17]


---
## 3. Moment of Inertia Tensor

The **moment of inertia tensor** $\mathbf{I}$ is a symmetric 3×3 matrix that encodes how the molecular mass is distributed in three-dimensional space. In the COM frame, it is defined as:

$$\mathbf{I} = \begin{pmatrix} I_{xx} & I_{xy} & I_{xz} \\ I_{yx} & I_{yy} & I_{yz} \\ I_{zx} & I_{zy} & I_{zz} \end{pmatrix}$$

The **diagonal elements** (moments of inertia) are:

$$I_{xx} = \sum_i m_i(y_i^2 + z_i^2), \quad I_{yy} = \sum_i m_i(x_i^2 + z_i^2), \quad I_{zz} = \sum_i m_i(x_i^2 + y_i^2)$$

The **off-diagonal elements** (products of inertia) are:

$$I_{xy} = I_{yx} = -\sum_i m_i x_i y_i, \quad I_{xz} = I_{zx} = -\sum_i m_i x_i z_i, \quad I_{yz} = I_{zy} = -\sum_i m_i y_i z_i$$

Note: these are computed in COM-shifted coordinates (primes dropped for clarity).

In [5]:
# Build the 3x3 inertia tensor
I = np.zeros((3, 3))

for m, (x, y, z) in zip(masses, coords_shifted):
    # Diagonal elements
    I[0, 0] += m * (y**2 + z**2)   # I_xx
    I[1, 1] += m * (x**2 + z**2)   # I_yy
    I[2, 2] += m * (x**2 + y**2)   # I_zz
    # Off-diagonal elements (products of inertia, negative by convention)
    I[0, 1] -= m * x * y            # I_xy
    I[0, 2] -= m * x * z            # I_xz
    I[1, 2] -= m * y * z            # I_yz

# Tensor is symmetric: fill lower triangle
I[1, 0] = I[0, 1]
I[2, 0] = I[0, 2]
I[2, 1] = I[1, 2]

print("Moment of Inertia Tensor (amu·Å²):")
print()
labels = ['x', 'y', 'z']
header = "        " + "  ".join(f"   {l:>10}" for l in labels)
print(header)
for i, row in enumerate(I):
    print(f"  {labels[i]}   " + "  ".join(f"{v:12.4f}" for v in row))

Moment of Inertia Tensor (amu·Å²):

                    x              y              z
  x       460.8155     -223.3967     -188.3295
  y      -223.3967      835.7129     -161.9181
  z      -188.3295     -161.9181     1000.5427


---
## 4. Principal Moments of Inertia

Because $\mathbf{I}$ is real and symmetric, it can be **diagonalized**, which is equivalent to rotating it to a special coordinate frame called the **principal axis frame**. The eigenvalues of $\mathbf{I}$ are the **principal moments of inertia** $I_A \leq I_B \leq I_C$:

$$\mathbf{I} \, \mathbf{u}_k = I_k \, \mathbf{u}_k, \quad k \in \{A, B, C\}$$

where $\mathbf{u}_k$ are the corresponding unit eigenvectors (the principal axes). In the principal axis frame, all off-diagonal elements vanish:

$$\mathbf{I}_{\text{diag}} = \begin{pmatrix} I_A & 0 & 0 \\ 0 & I_B & 0 \\ 0 & 0 & I_C \end{pmatrix}, \quad I_A \leq I_B \leq I_C$$

The convention is $A \geq B \geq C$ for **rotational constants** (see Section 5), which corresponds to $I_A \leq I_B \leq I_C$.

In [6]:
# Diagonalize the inertia tensor
# eigvalsh uses the symmetric structure for numerical stability
eigenvalues, eigenvectors = np.linalg.eigh(I)
eigenvalues.sort()   # Sort ascending: I_A ≤ I_B ≤ I_C

print("Principal Moments of Inertia (amu·Å²):")
for label, Ip in zip(['I_A', 'I_B', 'I_C'], eigenvalues):
    print(f"  {label} = {Ip:.6f}")

Principal Moments of Inertia (amu·Å²):
  I_A = 276.883859
  I_B = 915.772771
  I_C = 1104.414515


---
## 5. Rotational Constants

In microwave and rotational spectroscopy, the **rotational constant** $B$ for a diatomic (or, more generally, one principal axis) is:

$$B = \frac{h}{8\pi^2 c \, I}$$

where:
- $h$ = Planck's constant ($6.626 \times 10^{-34}$ J·s)
- $c$ = speed of light in cm/s ($2.998 \times 10^{10}$ cm/s)
- $I$ = moment of inertia in kg·m²

This gives $B$ in **wavenumbers** (cm⁻¹), the standard spectroscopic unit.

For a polyatomic molecule we compute three constants $A \geq B \geq C$ from the three principal moments $I_A \leq I_B \leq I_C$:

$$A = \frac{h}{8\pi^2 c \, I_A}, \quad B = \frac{h}{8\pi^2 c \, I_B}, \quad C = \frac{h}{8\pi^2 c \, I_C}$$

**Unit conversion:** The principal moments from NumPy are in amu·Å². To convert to kg·m²:

$$I_{\text{SI}} = I_{\text{amu·Å}^2} \times \underbrace{1.661 \times 10^{-27}}_{\text{kg/amu}} \times \underbrace{(10^{-10})^2}_{\text{m}^2/\text{Å}^2}$$

In [7]:
# Conversion factor: amu·Å² → kg·m²
conv = AMU_TO_KG * (ANG_TO_M**2)

# Rotational constants A, B, C in cm⁻¹
rot_consts = [H / (8 * np.pi**2 * C_LIGHT * (Ip * conv)) for Ip in eigenvalues]

print(f"{'Constant':<10} {'Computed (cm⁻¹)':<20} {'Psi4 ref (cm⁻¹)':<20} {'Match?'}")
print("-" * 65)
for label, calc, ref in zip(['A', 'B', 'C'], rot_consts, rot_psi4):
    match = np.isclose(calc, ref, atol=1e-4)
    print(f"  {label:<8} {calc:<20.5f} {ref:<20.5f} {match}")

Constant   Computed (cm⁻¹)      Psi4 ref (cm⁻¹)      Match?
-----------------------------------------------------------------
  A        0.06088              0.06088              True
  B        0.01841              0.01841              True
  C        0.01526              0.01526              True


---
## 6. Dipole Moment Transformation Under an Origin Shift

The **electric dipole moment** of a molecule with total charge $Q$ depends on the choice of origin. When the origin is shifted by a vector $\mathbf{d}$ (i.e., from the arbitrary lab frame to the COM), the dipole transforms as:

$$\boldsymbol{\mu}_{\text{new}} = \boldsymbol{\mu}_{\text{old}} - Q \, \mathbf{d}$$

where:
- $\boldsymbol{\mu}_{\text{old}}$ = dipole in the original frame (e·bohr)
- $Q$ = total charge of the molecule (in units of $e$)
- $\mathbf{d}$ = shift vector = COM position in **bohr** (atomic units match the dipole units)

> **Why does this matter?** For a **neutral** molecule ($Q = 0$), the dipole is origin-independent. But for a charged molecule (cation or anion), the dipole moment changes with the choice of origin. The physically meaningful value is the one computed in the **COM frame**.

The unit conversion for the shift vector:
$$\mathbf{d}_{\text{bohr}} = \mathbf{R}_{\text{COM}}[\text{Å}] \times \frac{1}{0.52918 \text{ Å/bohr}}$$

In [8]:
# Convert the COM shift vector from Å to bohr (atomic units)
com_bohr = com * ANG_TO_BOHR

# Apply the dipole transformation formula
# μ_new = μ_old - Q * d_bohr
dipole_shifted_au = dipole_unshifted_au - (TOTAL_CHARGE * com_bohr)

print("Dipole moment components (atomic units: e·bohr)")
print()
print(f"{'Component':<12} {'Original frame':<20} {'COM frame (calc)':<22} {'COM frame (Psi4)':<22} {'Match?'}")
print("-" * 90)
for i, comp in enumerate(['X', 'Y', 'Z']):
    match = np.isclose(dipole_shifted_au[i], dipole_shifted_psi4_au[i], atol=1e-4)
    print(f"  Dipole {comp}    {dipole_unshifted_au[i]:<20.7f} {dipole_shifted_au[i]:<22.7f} {dipole_shifted_psi4_au[i]:<22.7f} {match}")

# Dipole magnitude and Debye conversion
AU_TO_DEBYE = 2.5417464519
mag_calc  = np.linalg.norm(dipole_shifted_au) * AU_TO_DEBYE
mag_psi4  = np.linalg.norm(dipole_shifted_psi4_au) * AU_TO_DEBYE
print()
print(f"Dipole magnitude (Debye): computed = {mag_calc:.4f},  Psi4 = {mag_psi4:.4f}")

Dipole moment components (atomic units: e·bohr)

Component    Original frame       COM frame (calc)       COM frame (Psi4)       Match?
------------------------------------------------------------------------------------------
  Dipole X    -0.1208851           -1.9326734             -1.9326733             True
  Dipole Y    -0.2497723           1.0386462              1.0386462              True
  Dipole Z    -0.0089932           1.2694641              1.2694640              True

Dipole magnitude (Debye): computed = 6.4430,  Psi4 = 6.4430


---
## 7. Summary

Let's print a clean summary of all results.

In [9]:
print("=" * 70)
print("  MOLECULAR ROTATIONAL ANALYSIS SUMMARY")
print("  Molecule: 4-Bromonitrobenzene cation (C₆H₄BrNO₂⁺)")
print("=" * 70)
print()
print(f"Total mass:          {total_mass:.6f} amu")
print(f"Center of mass (Å):  x = {com[0]:.6f},  y = {com[1]:.6f},  z = {com[2]:.6f}")
print()
print("Moment of Inertia Tensor (amu·Å²):")
for i, row in enumerate(I):
    print("  " + "  ".join(f"{v:10.3f}" for v in row))
print()
print("Principal Moments (amu·Å²):")
for lbl, Ip in zip(['I_A', 'I_B', 'I_C'], eigenvalues):
    print(f"  {lbl} = {Ip:.4f}")
print()
print("Rotational Constants (cm⁻¹):")
for lbl, rc in zip(['A', 'B', 'C'], rot_consts):
    print(f"  {lbl} = {rc:.5f}")
print()
print("Dipole Moment in COM frame (e·bohr):")
for lbl, d in zip(['X', 'Y', 'Z'], dipole_shifted_au):
    print(f"  μ_{lbl} = {d:.6f}")
print(f"  |μ|  = {np.linalg.norm(dipole_shifted_au)*AU_TO_DEBYE:.4f} Debye")
print()
print("All values verified against Psi4 reference data. ✓")
print("=" * 70)

  MOLECULAR ROTATIONAL ANALYSIS SUMMARY
  Molecule: 4-Bromonitrobenzene cation (C₆H₄BrNO₂⁺)

Total mass:          201.950366 amu
Center of mass (Å):  x = 0.958757,  y = -0.681802,  z = -0.676530

Moment of Inertia Tensor (amu·Å²):
     460.815    -223.397    -188.329
    -223.397     835.713    -161.918
    -188.329    -161.918    1000.543

Principal Moments (amu·Å²):
  I_A = 276.8839
  I_B = 915.7728
  I_C = 1104.4145

Rotational Constants (cm⁻¹):
  A = 0.06088
  B = 0.01841
  C = 0.01526

Dipole Moment in COM frame (e·bohr):
  μ_X = -1.932673
  μ_Y = 1.038646
  μ_Z = 1.269464
  |μ|  = 6.4430 Debye

All values verified against Psi4 reference data. ✓


---
## Appendix: Key Equations Reference

| Quantity | Formula | Units |
|---|---|---|
| Center of mass | $\mathbf{R}_{\text{COM}} = \frac{\sum_i m_i \mathbf{r}_i}{M}$ | Å (or same as input) |
| Diagonal inertia elements | $I_{xx} = \sum_i m_i(y_i^2 + z_i^2)$ | amu·Å² |
| Off-diagonal inertia elements | $I_{xy} = -\sum_i m_i x_i y_i$ | amu·Å² |
| Principal moments | Eigenvalues of $\mathbf{I}$ | amu·Å² |
| Rotational constant | $B = \frac{h}{8\pi^2 c I}$ | cm⁻¹ |
| Dipole under origin shift | $\boldsymbol{\mu}' = \boldsymbol{\mu} - Q\mathbf{d}$ | e·bohr |
| Unit conversion | $I_{\text{kg·m}^2} = I_{\text{amu·Å}^2} \times 1.661\times10^{-27} \times 10^{-20}$ | — |

### Physical Interpretation

- A **larger** moment of inertia means a **smaller** rotational constant and more closely spaced rotational energy levels.
- For a **planar** molecule, the out-of-plane moment satisfies $I_C \approx I_A + I_B$ (the **inertial defect** is close to zero).
- The dipole origin-dependence is unique to **charged** species. For neutral molecules, the dipole is an intrinsic molecular property independent of origin choice.